In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CSV_PATH = "/content/drive/MyDrive/NN Multihead 2025/DatasetMTH_coeffs.csv"

Mounted at /content/drive


In [5]:
# =======================
# Short, clean PyTorch regressor: var1..var10 -> Fcr
# + time training/predict + EARLY STOPPING
# =======================
import os, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------- config --------
VAR_COLS = [f"var{i}" for i in range(1, 11)]
FCR_COL  = "Fcr"

ENC = [156, 105]   # 18,307 params
ACT = "relu"
DROPOUT = 0.0
BATCHNORM = False

EPOCHS = 200
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4

TEST_SIZE = 0.2
VAL_SIZE  = 0.1
SEED = 42

# Early stopping
ES_PATIENCE  = 8
ES_MIN_DELTA = 0.5e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
pin_memory = device.startswith("cuda")

# -------- utils --------
def seed_everything(seed=42, deterministic=False):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def sync():
    if device.startswith("cuda"):
        torch.cuda.synchronize()

def get_act(name: str) -> nn.Module:
    name = name.lower()
    table = {
        "relu": nn.ReLU(inplace=True),
        "gelu": nn.GELU(),
        "silu": nn.SiLU(inplace=True),
        "tanh": nn.Tanh(),
        "leaky_relu": nn.LeakyReLU(0.1, inplace=True),
        "elu": nn.ELU(inplace=True),
        "softplus": nn.Softplus(),
    }
    if name not in table: raise ValueError(f"Unknown activation: {name}")
    return table[name]

def make_loader(X, y, bs, shuffle, pin_memory):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    ds = TensorDataset(X, y)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=False,
                      pin_memory=pin_memory, num_workers=0)

def count_params(m):
    p_all = sum(p.numel() for p in m.parameters())
    p_tr  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return p_all, p_tr

# -------- model --------
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, enc, act="relu", dropout=0.0, batchnorm=False):
        super().__init__()
        layers, prev = [], in_dim
        for h in enc:
            layers.append(nn.Linear(prev, h))
            if batchnorm: layers.append(nn.BatchNorm1d(h))
            layers.append(get_act(act))
            if dropout and dropout > 0: layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)

@torch.no_grad()
def eval_mse(model, loader):
    model.eval()
    sse = 0.0; n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        sse += torch.sum((pred - y) ** 2).item()
        n += X.size(0)
    return sse / max(n, 1)

# -------- train --------
seed_everything(SEED, deterministic=False)

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

X = df[VAR_COLS].values.astype(np.float32)
F = df[FCR_COL].astype(np.float32).values.reshape(-1, 1)

X_tr, X_te, F_tr, F_te = train_test_split(X, F, test_size=TEST_SIZE, random_state=SEED)
X_tr, X_va, F_tr, F_va = train_test_split(X_tr, F_tr, test_size=VAL_SIZE, random_state=SEED)

scX, scF = StandardScaler(), StandardScaler()
X_tr_s = scX.fit_transform(X_tr); X_va_s = scX.transform(X_va); X_te_s = scX.transform(X_te)
F_tr_s = scF.fit_transform(F_tr); F_va_s = scF.transform(F_va); F_te_s = scF.transform(F_te)

dl_tr = make_loader(X_tr_s, F_tr_s, BATCH_SIZE, shuffle=True,  pin_memory=pin_memory)
dl_va = make_loader(X_va_s, F_va_s, BATCH_SIZE, shuffle=False, pin_memory=pin_memory)
dl_te = make_loader(X_te_s, F_te_s, BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

model = MLPRegressor(in_dim=len(VAR_COLS), enc=ENC, act=ACT, dropout=DROPOUT, batchnorm=BATCHNORM).to(device)
p_all, p_tr = count_params(model)
print(f"Params: total={p_all:,} trainable={p_tr:,} | device={device}")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

best = float("inf")
best_state = None
bad = 0  # early stop counter

sync(); t_train0 = time.perf_counter()

for ep in range(1, EPOCHS + 1):
    sync(); t0 = time.perf_counter()

    model.train()
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    va_mse = eval_mse(model, dl_va)

    # --- Early Stopping on val_mse ---
    if va_mse < best - ES_MIN_DELTA:
        best = va_mse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    sync(); t1 = time.perf_counter()
    print(f"Epoch {ep:03d} | val_mse={va_mse:.6f} | val_rmse={va_mse**0.5:.6f} | time={t1-t0:.3f}s | bad={bad}/{ES_PATIENCE}")

    if bad >= ES_PATIENCE:
        print(f"[EarlyStop] epoch {ep} | best_val_mse={best:.6f}")
        break

sync(); t_train1 = time.perf_counter()
epochs_ran = ep
print(f"\nTRAIN TIME (total): {t_train1-t_train0:.3f}s | epochs_ran={epochs_ran} | avg/epoch={(t_train1-t_train0)/epochs_ran:.3f}s")

if best_state is not None:
    model.load_state_dict(best_state)

te_mse = eval_mse(model, dl_te)
print(f"TEST: mse={te_mse:.6f} rmse={te_mse**0.5:.6f}")

# -------- inference (returns original scale + predict time) --------
@torch.no_grad()
def predict_fcr(var1_to_var10):
    x = np.array(var1_to_var10, dtype=np.float32).reshape(1, -1)
    x_s = scX.transform(x).astype(np.float32)
    xt  = torch.tensor(x_s).to(device)

    # tiny warm-up helps stable timing on GPU (optional)
    # _ = model(xt); sync()

    sync(); t0 = time.perf_counter()
    y_s = model(xt).cpu().numpy()
    sync(); t1 = time.perf_counter()

    y = scF.inverse_transform(y_s)[0, 0]
    return float(y), (t1 - t0)

@torch.no_grad()
def test_time_fullset_forward_only(model, loader, device="cpu", n_warmup_batches=5):
    model.eval()
    is_cuda = str(device).startswith("cuda")

    # warmup vài batch cho ổn định timing (GPU)
    it = iter(loader)
    for _ in range(n_warmup_batches):
        try:
            Xb, _ = next(it)
        except StopIteration:
            break
        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)
    if is_cuda:
        torch.cuda.synchronize()

    # timed full pass
    t0 = time.perf_counter()
    for Xb, _ in loader:
        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)
    if is_cuda:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    return t1 - t0

t_test_forward = test_time_fullset_forward_only(model, dl_te, device=device, n_warmup_batches=5)
print(f"TEST TIME (forward-only, full test set): {t_test_forward:.6f}s | batch_size={BATCH_SIZE} | n_test={len(X_te_s):,}")

# Example:
# y, dt = predict_fcr([0,0,0,0,0,0,0,0,0,0])
# print("Fcr=", y, "| predict_time=", dt*1e3, "ms")


Params: total=18,307 trainable=18,307 | device=cpu
Epoch 001 | val_mse=0.001608 | val_rmse=0.040104 | time=6.856s | bad=0/8
Epoch 002 | val_mse=0.000607 | val_rmse=0.024634 | time=7.594s | bad=0/8
Epoch 003 | val_mse=0.000469 | val_rmse=0.021655 | time=6.834s | bad=0/8
Epoch 004 | val_mse=0.000337 | val_rmse=0.018365 | time=7.488s | bad=0/8
Epoch 005 | val_mse=0.000257 | val_rmse=0.016028 | time=7.188s | bad=0/8
Epoch 006 | val_mse=0.000149 | val_rmse=0.012220 | time=7.127s | bad=0/8
Epoch 007 | val_mse=0.000090 | val_rmse=0.009480 | time=7.892s | bad=0/8
Epoch 008 | val_mse=0.000060 | val_rmse=0.007742 | time=6.506s | bad=0/8
Epoch 009 | val_mse=0.000140 | val_rmse=0.011847 | time=7.892s | bad=1/8
Epoch 010 | val_mse=0.000126 | val_rmse=0.011210 | time=6.534s | bad=2/8
Epoch 011 | val_mse=0.000061 | val_rmse=0.007824 | time=7.688s | bad=3/8
Epoch 012 | val_mse=0.000055 | val_rmse=0.007449 | time=6.328s | bad=4/8
Epoch 013 | val_mse=0.000105 | val_rmse=0.010243 | time=7.918s | bad=5/8


In [ ]:
CSV_PATH = "/content/drive/MyDrive/NN Multihead 2025/DatasetMTH_coeffs.csv"

In [6]:
# =======================
# Clean multi-output PyTorch regressor: var1..var10 -> [L0, L1, L2]
# - StandardScaler for X and Y(train-only)
# - Configurable: ENC, ACT, DROPOUT, BATCHNORM
# - Loss: MSE (3 outputs)
# - Prints: training time, test time (full loader), predict time (single sample)
# =======================
import random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------- config --------

VAR_COLS = [f"var{i}" for i in range(1, 11)]
Y_COLS   = ["L0", "L1", "L2"]   # <-- 3 output columns

ENC = [128,128]               # you can change
ACT = "relu"
DROPOUT = 0.0
BATCHNORM = False

EPOCHS = 200
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4

TEST_SIZE = 0.2
VAL_SIZE  = 0.1
SEED = 42

# Early stopping
ES_PATIENCE  = 8
ES_MIN_DELTA = 0.5e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
pin_memory = device.startswith("cuda")

# -------- utils --------
def seed_everything(seed=42, deterministic=False):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def sync():
    if device.startswith("cuda"):
        torch.cuda.synchronize()

def get_act(name: str) -> nn.Module:
    name = name.lower()
    table = {
        "relu": nn.ReLU(inplace=True),
        "gelu": nn.GELU(),
        "silu": nn.SiLU(inplace=True),
        "tanh": nn.Tanh(),
        "leaky_relu": nn.LeakyReLU(0.1, inplace=True),
        "elu": nn.ELU(inplace=True),
        "softplus": nn.Softplus(),
    }
    if name not in table: raise ValueError(f"Unknown activation: {name}")
    return table[name]

def make_loader(X, y, bs, shuffle, pin_memory):
    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)
    ds = TensorDataset(X, y)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=False,
                      pin_memory=pin_memory, num_workers=0)

def count_params(m):
    p_all = sum(p.numel() for p in m.parameters())
    p_tr  = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return p_all, p_tr

# -------- model --------
class MLPRegressor(nn.Module):
    def __init__(self, in_dim, out_dim, enc, act="relu", dropout=0.0, batchnorm=False):
        super().__init__()
        layers, prev = [], in_dim
        for h in enc:
            layers.append(nn.Linear(prev, h))
            if batchnorm: layers.append(nn.BatchNorm1d(h))
            layers.append(get_act(act))
            if dropout and dropout > 0: layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)

@torch.no_grad()
def eval_mse(model, loader):
    model.eval()
    sse = 0.0; n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        sse += torch.sum((pred - y) ** 2).item()  # sum over batch & dims
        n += X.size(0)
    return sse / max(n, 1)  # mean over samples (not dims)

@torch.no_grad()
def eval_mse_per_dim(model, loader, y_dim=3):
    model.eval()
    sse = np.zeros((y_dim,), dtype=np.float64)
    n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        pred = model(X)
        diff2 = (pred - y) ** 2
        sse += diff2.sum(dim=0).detach().cpu().numpy()
        n += X.size(0)
    mse = sse / max(n, 1)
    rmse = np.sqrt(mse)
    return mse, rmse

# -------- train --------
seed_everything(SEED, deterministic=False)

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

need = VAR_COLS + Y_COLS
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df = df[need].replace([np.inf, -np.inf], np.nan).dropna()

X = df[VAR_COLS].values.astype(np.float32)
Y = df[Y_COLS].values.astype(np.float32)     # (N, 3)

# split: train/test then train/val
X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=TEST_SIZE, random_state=SEED)
X_tr, X_va, Y_tr, Y_va = train_test_split(X_tr, Y_tr, test_size=VAL_SIZE, random_state=SEED)

# scale train-only (separate scalers)
scX, scY = StandardScaler(), StandardScaler()
X_tr_s = scX.fit_transform(X_tr); X_va_s = scX.transform(X_va); X_te_s = scX.transform(X_te)
Y_tr_s = scY.fit_transform(Y_tr); Y_va_s = scY.transform(Y_va); Y_te_s = scY.transform(Y_te)

dl_tr = make_loader(X_tr_s, Y_tr_s, BATCH_SIZE, shuffle=True,  pin_memory=pin_memory)
dl_va = make_loader(X_va_s, Y_va_s, BATCH_SIZE, shuffle=False, pin_memory=pin_memory)
dl_te = make_loader(X_te_s, Y_te_s, BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

model = MLPRegressor(in_dim=len(VAR_COLS), out_dim=3, enc=ENC, act=ACT, dropout=DROPOUT, batchnorm=BATCHNORM).to(device)
p_all, p_tr = count_params(model)
print(f"Params: total={p_all:,} trainable={p_tr:,} | device={device}")

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.MSELoss()

best = float("inf")
best_state = None
bad = 0

sync(); t_train0 = time.perf_counter()

for ep in range(1, EPOCHS + 1):
    sync(); t0 = time.perf_counter()

    model.train()
    for Xb, yb in dl_tr:
        Xb, yb = Xb.to(device), yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)     # MSE over 3 dims
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

    va_mse = eval_mse(model, dl_va)

    # Early stopping on val_mse
    if va_mse < best - ES_MIN_DELTA:
        best = va_mse
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1

    sync(); t1 = time.perf_counter()
    print(f"Epoch {ep:03d} | val_mse={va_mse:.6f} | val_rmse~={(va_mse**0.5):.6f} | time={t1-t0:.3f}s | bad={bad}/{ES_PATIENCE}")

    if bad >= ES_PATIENCE:
        print(f"[EarlyStop] epoch {ep} | best_val_mse={best:.6f}")
        break

sync(); t_train1 = time.perf_counter()
epochs_ran = ep
print(f"\nTRAIN TIME: {t_train1-t_train0:.3f}s | epochs_ran={epochs_ran} | avg/epoch={(t_train1-t_train0)/epochs_ran:.3f}s")

if best_state is not None:
    model.load_state_dict(best_state)

# -------- test time (full loader) --------
sync(); t_test0 = time.perf_counter()
te_mse = eval_mse(model, dl_te)
sync(); t_test1 = time.perf_counter()

mse_dim, rmse_dim = eval_mse_per_dim(model, dl_te, y_dim=3)
print(f"\nTEST: mse={te_mse:.6f} rmse~={te_mse**0.5:.6f} | eval_time_mse={t_test1-t_test0:.3f}s")
print(f"TEST per-dim RMSE: L0={rmse_dim[0]:.6f} | L1={rmse_dim[1]:.6f} | L2={rmse_dim[2]:.6f}")

# -------- curve rmse (50 points) --------
Nw = 50
W_GRID = np.linspace(0.0, 1.0, Nw).astype(np.float32)

@torch.no_grad()
def eval_curve_rmse50(model, loader, scY: StandardScaler, W_GRID, device="cpu", return_per_point=False):
    model.eval()
    y_true_s_list, y_pred_s_list = [], []

    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        y_pred_s = model(Xb)  # scaled
        y_true_s_list.append(yb.detach().cpu().numpy())
        y_pred_s_list.append(y_pred_s.detach().cpu().numpy())

    y_true_s = np.vstack(y_true_s_list)  # (N,3)
    y_pred_s = np.vstack(y_pred_s_list)  # (N,3)

    # inverse -> original scale
    y_true = scY.inverse_transform(y_true_s).astype(np.float64)  # (N,3)
    y_pred = scY.inverse_transform(y_pred_s).astype(np.float64)  # (N,3)

    W = W_GRID.reshape(1, -1).astype(np.float64)  # (1,50)

    # curve reconstruction
    curve_true = y_true[:, 0:1] + y_true[:, 1:2] * W + y_true[:, 2:3] * (W**2)  # (N,50)
    curve_pred = y_pred[:, 0:1] + y_pred[:, 1:2] * W + y_pred[:, 2:3] * (W**2)  # (N,50)

    diff2 = (curve_pred - curve_true) ** 2
    mse_curve = float(diff2.mean())               # mean over N*50
    rmse_curve = float(np.sqrt(mse_curve))

    if return_per_point:
        mse_per_point = diff2.mean(axis=0)        # (50,)
        rmse_per_point = np.sqrt(mse_per_point)
        return rmse_curve, rmse_per_point.astype(np.float32)

    return rmse_curve

# compute + print
rmse_curve50 = eval_curve_rmse50(model, dl_te, scY, W_GRID, device=device)
print(f"TEST: RMSE_Curve50={rmse_curve50:.6f}")

# -------- inference (returns original scale + predict time) --------
@torch.no_grad()
def test_time_fullset_forward_only(model, loader, device="cpu", n_warmup_batches=5):
    model.eval()
    is_cuda = str(device).startswith("cuda")

    # warmup
    it = iter(loader)
    for _ in range(n_warmup_batches):
        try:
            Xb, _ = next(it)
        except StopIteration:
            break
        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)
    if is_cuda:
        torch.cuda.synchronize()

    # timed full pass (forward-only)
    t0 = time.perf_counter()
    for Xb, _ in loader:
        Xb = Xb.to(device, non_blocking=True)
        _ = model(Xb)
    if is_cuda:
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    return t1 - t0
t_test_forward = test_time_fullset_forward_only(model, dl_te, device=device, n_warmup_batches=5)
print(f"TEST TIME (forward-only, full test set): {t_test_forward:.6f}s | batch_size={BATCH_SIZE} | n_test={len(X_te_s):,}")


# Example:
# pred, dt = predict_L012([0,0,0,0,0,0,0,0,0,0])
# print(pred, "| predict_time(ms)=", dt*1e3)


Params: total=18,307 trainable=18,307 | device=cpu
Epoch 001 | val_mse=0.005450 | val_rmse~=0.073824 | time=7.766s | bad=0/8
Epoch 002 | val_mse=0.001991 | val_rmse~=0.044617 | time=6.476s | bad=0/8
Epoch 003 | val_mse=0.001387 | val_rmse~=0.037238 | time=7.959s | bad=0/8
Epoch 004 | val_mse=0.000644 | val_rmse~=0.025385 | time=18.326s | bad=0/8
Epoch 005 | val_mse=0.000955 | val_rmse~=0.030909 | time=9.314s | bad=1/8
Epoch 006 | val_mse=0.000457 | val_rmse~=0.021380 | time=6.489s | bad=0/8
Epoch 007 | val_mse=0.000273 | val_rmse~=0.016508 | time=7.889s | bad=0/8
Epoch 008 | val_mse=0.000199 | val_rmse~=0.014101 | time=7.179s | bad=0/8
Epoch 009 | val_mse=0.000483 | val_rmse~=0.021973 | time=7.522s | bad=1/8
Epoch 010 | val_mse=0.000312 | val_rmse~=0.017655 | time=6.456s | bad=2/8
Epoch 011 | val_mse=0.000176 | val_rmse~=0.013251 | time=7.877s | bad=0/8
Epoch 012 | val_mse=0.000136 | val_rmse~=0.011650 | time=6.354s | bad=0/8
Epoch 013 | val_mse=0.000110 | val_rmse~=0.010477 | time=7.7